In [ ]:
# LABIST- PEAKS AITAMA CONNECTIONIT TEHA (in lab task 1.4)
import psycopg2

PG_CONN = dict(host="postgres", port=5432, dbname="sourcedb", user="cdc_user", password="cdc_pass")

def pg_execute(sql, fetch=False):
    conn = psycopg2.connect(**PG_CONN)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql)
    result = cur.fetchall() if fetch else None
    cur.close()
    conn.close()
    return result

ver = pg_execute("SELECT version();", fetch=True)
print(f"PostgreSQL: {ver[0][0][:60]}...")

wal = pg_execute("SHOW wal_level;", fetch=True)
print(f"wal_level = {wal[0][0]}")
assert wal[0][0] == "logical", "wal_level must be 'logical' for CDC!"

In [ ]:
pg_execute("SELECT * FROM customers ORDER BY id;", fetch=True)

In [ ]:
# MINU VIIS TEHA 1.4 kasutades POWERSHELLIS
"""
$body = @{
>>     name = "cdc-connector"
>>     config = @{
>>         "connector.class" = "io.debezium.connector.postgresql.PostgresConnector"
>>         "database.hostname" = "postgres"
>>         "database.port" = "5432"
>>         "database.user" = "cdc_user"
>>         "database.password" = "admin"
>>         "database.dbname" = "sourcedb"
>>         "topic.prefix" = "dbserver1"
>>         "table.include.list" = "public.customers,public.drivers"
>>         "plugin.name" = "pgoutput"
>>         "slot.name" = "debezium_slot"
>>         "publication.name" = "dbz_publication"
>>     }
>> } | ConvertTo-Json -Depth 5
"""

"""
Invoke-RestMethod `
>>   -Uri "http://localhost:8083/connectors" `
>>   -Method Post `
>>   -ContentType "application/json" `
>>   -Body $body
"""

In [15]:
# ei tea kas see päriselt vajalik, lihtsalt labist võetud
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "psycopg2-binary", "kafka-python-ng"])
print("requests + psycopg2 + kafka-python-ng ready")

requests + psycopg2 + kafka-python-ng ready


In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import os

spark = (SparkSession.builder 
  .appName("CDC-Bronze") 
  .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
  .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") 
  .config("spark.sql.catalog.lakehouse.type", "rest") 
  .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
  .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")  
  .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
  .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
  .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
  .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
  .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
  .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
  .config("spark.sql.defaultCatalog", "lakehouse") 
  .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}")

Spark 4.1.0


In [9]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")

DataFrame[]

In [10]:
raw = (spark.read
  .format("kafka") 
  .option("kafka.bootstrap.servers", "kafka:9092") 
  .option("subscribe", "dbserver1.public.customers") 
  .option("startingOffsets", "earliest") 
  .load())

In [11]:
from pyspark.sql import functions as F


# Filter out tombstone records (null value) first
raw_filtered = raw.filter(F.col("value").isNotNull())


bronze_df = raw_filtered.select(
  F.col("topic"),
  F.col("partition").alias("kafka_partition"),
  F.col("offset").alias("kafka_offset"),
  F.col("timestamp").alias("kafka_timestamp"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.op").alias("op"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.ts_ms").cast("long").alias("ts_ms"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.id").cast("int").alias("after_id"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.name").alias("after_name"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.email").alias("after_email"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.after.country").alias("after_country"),
  F.get_json_object(F.col("value").cast("string"), "$.payload.before.id").cast("int").alias("before_id"),
)

In [12]:
bronze_df.writeTo("lakehouse.cdc.bronze_customers").createOrReplace() # change to append() in actual project